<a id="Snowflake_Feature_Store_L3"></a>
# Snowflake Feature Store Part 3


<a id="topics"></a>
### 1.1 Topics in this lesson


1. [Snowflake Feature Store](#Snowflake_Feature_Store_L3)
    1. [Topics in this Lesson](#topics)
    1. [Initial setup](#Initial_setup)
    1. [Environment](#Environment)
    
1. [Operationalization](#Operationalization)  
    1. [Setting up production feature store and dataframes](#Setting_up_production)  
    1. [Rerun preprocessing](#Rerun_preprocessing)  
    1. [Create scheduled inference pipeline](#Create_scheduled_inference_pipeline) 
    1. [Create & Register Inference-FeatureView to run scheduled Inference](#Create_Register)  
    
    

<a id="Initial_setup"></a>
### 1.2 Initial setup

#### Connect and create a `Session` 
1. Import required libraries.
1. Create a `Session` to connect to Snowflake.  
<br />  
    > &#10071; Success requires that you have already completed the key pair authentication exercise.
1. Set context items for this module.

Run the following cell to connect to your Snowflake account. *You needn't edit anything in the following cell. Just run it.*

In [ ]:
# Run utils notebook
%run ../../utils/ds_utils_python.ipynb

# Connect to Snowflake and create a Session object named session
session = create_session()

In [ ]:
# Python packages
import os
from os import listdir
from os.path import isfile, join
import time
import json
import timeit
import numpy as np
import pandas as pd
import datetime
import ast      
from datetime import date, datetime 
from decimal import Decimal

# Snowpark
from snowflake.snowpark import Session, DataFrame, Window, WindowSpec

import snowflake.snowpark.functions as F
import snowflake.snowpark.types as T
from snowflake.snowpark.version import VERSION

# Snowflake Feature Store
from snowflake.ml.feature_store import (
    FeatureStore,
    FeatureView,
    Entity,
    CreationMode)

# Snowflake Model Registry
from snowflake.ml.registry import Registry
from snowflake.ml.utils import connection_params
from snowflake.ml._internal.utils import  identifier  

# K-Means clustering
from snowflake.ml.modeling.pipeline import Pipeline as sml_Pipeline
from snowflake.ml.modeling.preprocessing import MinMaxScaler as sml_MinMaxScaler
from snowflake.ml.modeling.cluster import KMeans as sml_KMeans

<a id="Environment"></a>
### 1.3 Environment

In [ ]:
# Let us set the defaults again:

# Create an empty DataFrame for reflecting upon our context items
context_df = session.create_dataframe([""]).to_df("")

# Retrieve the current username
from snowflake.snowpark.functions import current_user
username = (str(context_df
    .select(current_user())
    .collect()[0][0]
   )
)
print(f"The current user is: {username}")


# Roles
ds_role                 = 'DS_ROLE'
aa_role                 = 'training_role'

# Database
segmentation_database_base       = f'{username}_SEGMENTATION' 
segmentation_database        = f'{segmentation_database_base}_LIVE'

# Schemas
segmentation_training_schema     = 'TRAINING'
segmentation_scoring_schema      = 'SCORING'
segmentation_serving_schema      = 'SERVING'

# Set the Schema
segmentation_schema = segmentation_serving_schema

# Set  Environment
snowflake_environment = session.sql('SELECT current_user(), current_version()').collect()
session.sql(f'''use role {ds_role}''').collect()
session.sql(f'''use database {segmentation_database}''').collect()
session.sql(f'''use schema {segmentation_training_schema}''').collect()


# Create a Warehouse
warehouse_sz = 'MEDIUM'
warehouse_env = f'{username}_SEGMENTATION_WH'
session.sql(f'''use warehouse {warehouse_env}''').collect()
session.sql(f'''alter warehouse {warehouse_env} set warehouse_size = {warehouse_sz}''').collect()



# Current Environment Details
print('\nConnection Established with the following parameters:')
print(f'User                        : {snowflake_environment[0][0]}')
print(f'Role                        : {session.get_current_role()}')
print(f'Database                    : {session.get_current_database()}')
print(f'Schema                      : {session.get_current_schema()}')
print(f'Warehouse                   : {session.get_current_warehouse()}')
print(f'Snowflake version           : {snowflake_environment[0][1]}')

<a id="Operationalization"></a>
## 2. Operationalization

Now we are at the operations side. Here we will see how easy it is to replicate the training feature-engineering pipeline, created during model development, in the SERVING (Production) schema

The Segmentation use case is designed to emulate a data science pipeline to find clusters of customers based on aggregate features where the customers are grouped based on their spending behavior.

It involves creating subgroups of customers based on similar traits.

The input in this use case consists of order and return transaction data from a retail business.

In the production (SERVING) environment we will 
- re-create the FeatureViews on production data
- generate an Inference FeatureView that uses the saved model to perform incremental inference

<a id="Setting_up_production"></a>
### 2.1 Setting up production feature store and dataframes

In [ ]:
# First we set up everything that is needed. We create a  Feature Store, we open the Model Registry and we create several Snowpark tables
fs = (FeatureStore(
        session  = session, 
        database = segmentation_database, 
        name     = f"""_{segmentation_schema}_FEATURE_STORE""", 
        default_warehouse = warehouse_env, 
        creation_mode = CreationMode.CREATE_IF_NOT_EXIST)
     )

# This opens the model registry again
mr = Registry(session=session, database_name= segmentation_database, schema_name='_MODEL_REGISTRY')

# Tables
customer_tbl                     = '.'.join([segmentation_database, segmentation_schema,'CUSTOMER'])
line_item_tbl                    = '.'.join([segmentation_database, segmentation_schema,'LINEITEM'])
order_tbl                        = '.'.join([segmentation_database, segmentation_schema,'ORDERS'])
order_returns_tbl                = '.'.join([segmentation_database, segmentation_schema,'ORDER_RETURNS'])

# Snowpark Dataframe
customer_sdf               = session.table(customer_tbl)
line_item_sdf              = session.table(line_item_tbl)
order_sdf                  = session.table(order_tbl)
order_returns_sdf          = session.table(order_returns_tbl)

model_name = "SNOWFLAKEML_KMEANS_MODEL"

# Row Counts
print(f'''\nTABLE ROW_COUNTS IN {segmentation_schema}''')
print(customer_tbl, customer_sdf.count())
print(line_item_tbl, line_item_sdf.count())
print(order_tbl, order_sdf.count())
print(order_returns_tbl, order_returns_sdf.count())

<a id="Rerun_preprocessing"></a>
### 2.2 Rerun preprocessing

We can now rerun the exact same code that we lifted from our Development (TRAINING) process to recreate the Feature Engineering pipelines in production

> **&#128221; Note: This is the same code as in the previous notebook, but now a different schema is selected**

In [ ]:
### Create & Load Source Data
# Default replacement values for Null dates and decimal types
epoch_dt = date(year=1970, month=1, day=1)
decimal_zero = Decimal('0.0')

order_data =order_sdf
lineitem_data =line_item_sdf 
order_returns_data = order_returns_sdf

# Merge three dataframes
raw_data =  (lineitem_data
    .join(
        order_returns_data,
        (lineitem_data["LI_ORDER_ID"] == order_returns_data["OR_ORDER_ID"]) &
        (lineitem_data["LI_PRODUCT_ID"] == order_returns_data["OR_PRODUCT_ID"]),
        "left") \
    .join(
        order_data,
        order_returns_data["OR_ORDER_ID"] == order_data["O_ORDER_ID"],
        "inner") \
    .select(   "O_ORDER_ID",  "O_CUSTOMER_SK", "ORDER_TS", "WEEKDAY", "ORDER_DATE", "LI_PRODUCT_ID", "PRICE", "QUANTITY", "OR_RETURN_QUANTITY") \
    .fillna({ "O_ORDER_ID": 0, "O_CUSTOMER_SK": 0, "ORDER_DATE": epoch_dt, "PRICE": decimal_zero, "QUANTITY": 0, "OR_RETURN_QUANTITY": 0 })
            )

raw_data = raw_data[['O_ORDER_ID', 'O_CUSTOMER_SK', 'ORDER_DATE', 'LI_PRODUCT_ID', 'PRICE', 'QUANTITY', 'OR_RETURN_QUANTITY']]

print('''--- Created Source Data ---''')



### Create & Run Preprocessing Function 
# Calculate INVOICE_YEAR, ROW_PRICE and RETURN_ROW_PRICE
data = raw_data.with_columns(["INVOICE_YEAR",  "ROW_PRICE",  "RETURN_ROW_PRICE" ]
                        ,[ F.year(raw_data["ORDER_DATE"]),  raw_data["QUANTITY"] * raw_data["PRICE"],  raw_data["OR_RETURN_QUANTITY"] * raw_data["PRICE"]] )

# Generate Customer/Order level features : total-price, total-return-price, year of first order last-order-date
groups = data.groupBy("O_CUSTOMER_SK", "O_ORDER_ID").agg(
    F.sum(F.col("ROW_PRICE")).alias("ROW_PRICE"),
    F.sum(F.col("RETURN_ROW_PRICE")).alias("RETURN_ROW_PRICE"),
    F.min(F.col("INVOICE_YEAR")).alias("INVOICE_YEAR"),
    F.max(F.col("ORDER_DATE")).alias("LATEST_ORDER_DATE"))

# Calculate price RETURN RATIO per Customer
groups = groups.withColumn("RATIO", groups["RETURN_ROW_PRICE"] / groups["ROW_PRICE"])
ratio = groups.groupBy("O_CUSTOMER_SK").agg(F.avg(F.col("RATIO")).cast(T.FloatType()).alias("RETURN_RATIO"), 
                                            F.max(F.col("LATEST_ORDER_DATE")).alias("LATEST_ORDER_DATE")
                                            )

# Calculate average annual shopping FREQUENCY 
frequency_groups = groups.groupBy("O_CUSTOMER_SK", "INVOICE_YEAR").agg(F.count(F.col("O_ORDER_ID")).cast(T.FloatType()).alias("FREQUENCY"))
frequency = frequency_groups.groupBy("O_CUSTOMER_SK").agg(F.avg(F.col("FREQUENCY")).alias("FREQUENCY"))

# Merge FREQUENCY and RETURN_RATIO
preprocessed_data = frequency.join(ratio, on="O_CUSTOMER_SK")
print('''--- Created Preprocessed Data ---''')

# Customer entity
customer_entity = Entity(name="CUSTOMER", join_keys=["O_CUSTOMER_SK"],desc="Primary Key for CUSTOMER")
fs.register_entity(customer_entity)

### Create Preprocessing FeatureView from Preprocess Dataframe (SQL)
fv_name = "FV_PREPROCESSING"
fv_version = "V_1"
# Define descriptions for the FeatureView's Features.  These will be added as comments to the database object
preprocess_features_desc = { "FREQUENCY":"Average yearly order frequency",
                             "RETURN_RATIO":"Average of, Per Order Returns Ratio.  Per order returns ratio : total returns value / total order value" }
# Create Inference Feature View
try:
    # If FeatureView already exists just return the reference to it
    fv_preprocessing = fs.get_feature_view(name=fv_name,version=fv_version)
except:
    # Create the FeatureView instance
    fv_preprocessing_instance = FeatureView(
        name=fv_name, 
        entities=[customer_entity], 
        feature_df=preprocessed_data,      # <- We can use the snowpark dataframe as-is from our Python
        timestamp_col="LATEST_ORDER_DATE",
        refresh_freq="60 minute",           # <- specifying optional refresh_freq creates FeatureView as Dynamic Table, else created as View.
        desc="Features to support Segmentation").attach_feature_desc(preprocess_features_desc)

    # Register the FeatureView instance.  Creates  object in Snowflake
    fv_preprocessing = fs.register_feature_view(
        feature_view=fv_preprocessing_instance, 
        version=fv_version, 
        block=True
    )
    print(f"Feature View : {fv_name}_{fv_version} created in {segmentation_schema}")   
else:
    print(f"Feature View : {fv_name}_{fv_version} already created in {segmentation_schema}")

print('''---            DONE               ---''')

Notice again, that we created and registered a FeatureView called `FV_PREPROCESSING`

- This query contain a lot of transformations. We can check the query in the Dynamic Tables section in Snowsight. 

<a id="Create_scheduled_inference_pipeline"></a>
### 2.3 Create scheduled inference pipeline

We define our model inference function again, whcih takes as input
- a model version 
- Inference Dataframe

In [ ]:
def serve(inference_df, model) -> DataFrame:
    return model.run(inference_df, function_name="predict") # function name = The function name to run 

We define a dataframe that reads all the records from our feature engineering pipeline. When used within the FeatureView, the Dynamic Table that gets created, will incrementally process change data once the initial Dynamic Table has been created.

So 
1. the FeatureView `FV_PREPROCESSING_V_1` is created. 
2. It refers to `preprocessed_data`.
3. That data comes from 3 tables in the `SEGMENTATION_LIVE` database and the `SERVING` schema
4. Data from `SEGMENTATION_LIVE.SERVING` comes incrementally from the `SEGMENTATION.SERVING` 
5. Which was again to mimic live data into a production environment

In [ ]:
# Create an Inference Dataframe that reads from our feature-engineering pipeline
inference_input_sdf = fs.read_feature_view(fv_preprocessing)
inference_input_sdf.show()

In [ ]:
# Get latest version of the model
m = mr.get_model(model_name)
latest_version = m.show_versions().iloc[-1]['name']
mv = m.version(latest_version)

Here we will run the model over FeatureView. The `ModelVersion.run` returns the prediction data. It would be the same type dataframe as your input.

- Is a Snowpark Dataframe here

In [ ]:
# Test Inference process
inference_result_sdf = serve(inference_input_sdf, mv)
inference_result_sdf.sort(F.col('LATEST_ORDER_DATE').desc(), F.col('O_CUSTOMER_SK')).show()

**SNEAKPEAK** (as we discuss the Model Registry in a later lecture):

Notice that we can also see in the SQL output below how our model is packaged and called from SQL

It uses `!predict`:
`MODEL_VERSION_ALIAS!PREDICT(RETURN_RATIO, FREQUENCY) AS TMP_RESULT`

And it takes the data from the featureView V_1:

`AS (SELECT * FROM SEGMENTATION_LIVE._SERVING_FEATURE_STORE.FV_PREPROCESSING$V_1`

*This is also visible in the Query History in Snowsight*

In [ ]:
ind_sql = inference_result_sdf.queries['queries'][0]
ind_fmtd_sql = os.linesep.join(ind_sql.split(os.linesep)[:1000])
print(ind_fmtd_sql)

<a id="Create_Register"></a>
### 2.4 Create & Register Inference-FeatureView to run scheduled Inference

We then create a FeatureView that will compute Inference on incremental data in the feature engineering pipeline, keeping an up to date set of customer segments through time.

In [ ]:
## Create & Register Inference-FeatureView to run scheduled Inference
inf_fv_name = "FV_INFERENCE_RESULT"
inf_fv_version = "V_2"

inference_features_desc = { "FREQUENCY":"Average yearly order frequency",
                              "RETURN_RATIO":"Average of, Per Order Returns Ratio.  Per order returns ratio : total returns value / total order value", 
                              "RETURN_RATIO_MMS":f"Min/Max Scaled version of RETURN_RATIO using Model Registry ({segmentation_database}_MODEL_REGISTRY) Model ({mv.model_name}) Model-Version({mv.version_name}) Model Comment ({mv.comment})",
                              "FREQUENCY_MMS":f"Min/Max Scaled version of FREQUENCY using Model Registry ({segmentation_database}_MODEL_REGISTRY) Model ({mv.model_name}) Model-Version({mv.version_name})  Model Comment ({mv.comment}",
                              "CLUSTER":f"Kmeans Cluster for Customer Clustering Model (UC01) using Model Registry ({segmentation_database}_MODEL_REGISTRY) Model ({mv.model_name}) Model-Version({mv.version_name})  Model Comment ({mv.comment}"}

# In case it already exists
try:
   fv_inference_result = fs.get_feature_view(name= inf_fv_name, version= inf_fv_version)
except:
   fv_inference_result = FeatureView(
         name= inf_fv_name, 
         entities=[customer_entity], 
         feature_df=inference_result_sdf,
         desc="Inference Result from kmeans model").attach_feature_desc(inference_features_desc)
   
   fv_inference_result = fs.register_feature_view(
         feature_view=fv_inference_result, 
         version= inf_fv_version, 
         block=True
   )
   print(f"Inference Feature View : fv_inference_result_{inf_fv_version} created")   
else:
   print(f"Inference Feature View : fv_inference_result_{inf_fv_version} already created")
finally:
   fs_serving_fviews = fs.list_feature_views().filter(F.col("NAME") == inf_fv_name ).sort(F.col("VERSION").desc())
   fs_serving_fviews.show()  

In [ ]:
fv_inference_result

To make it slightly more readable:

<img src="../../images/snowflake_feature_view_query.png" alt="snowflake_feature_view_query" style="width:70%;display:block;margin-left:10%;" />




We can see that the `FV_INFERENCE_RESULT` FeatureView gets created from scoring the PREPROCESSING$V_1 FeatureView

In [ ]:
fv_inference_result.feature_df.sort(F.col("LATEST_ORDER_DATE").desc()).show(10)


### Conclusion

Now we've seen a production environment that gets 'live' data. From this environment FeatureViews were created. 

- One of them will refresh every 60 minutes, getting the new data.
- The second takes that data and scores it
- All automatic





# Clean up

In [ ]:
session.sql(f"""use role {aa_role}""").collect()

In [ ]:
session.sql(f"""drop database {segmentation_database_base} """).collect()

In [ ]:
session.sql(f"""drop database {segmentation_database} """).collect()

In [ ]:
session.sql(f"""drop warehouse {warehouse_env} """).collect()

In [ ]:
session.close()